In [3]:
import pandas as pd

In [ ]:
df_funding = pd.read_csv("../../data/fts_requirements_funding_global.csv")
df_funding_cluster = pd.read_csv("../../data/fts_requirements_funding_globalcluster_global.csv")


In [5]:
df_funding_cluster

,countryCode,id,name,code,startDate,endDate,year,clusterCode,cluster,requirements,funding,percentFunded
0,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,26480.0,Coordination and support services,24700000.0,1805186.0,7.0
1,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,3.0,Education,60040000.0,18138612.0,30.0
2,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,4.0,Emergency Shelter and NFI,160285975.0,5814964.0,4.0
3,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,6.0,Food Security,651101609.0,50221603.0,8.0
4,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,7.0,Health,190768456.0,30587554.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...
10625,ZWE,98,Humanitarian Crisis in Southern Africa 2002 - ...,CZWE0203,2002-07-01,2003-06-30,2002,26479.0,Multi-sector,557000.0,2244422.0,403.0
10626,ZWE,98,Humanitarian Crisis in Southern Africa 2002 - ...,CZWE0203,2002-07-01,2003-06-30,2002,10.0,Protection,1500000.0,NaN,NaN
10627,ZWE,98,Humanitarian Crisis in Southern Africa 2002 - ...,CZWE0203,2002-07-01,2003-06-30,2002,11.0,Water Sanitation Hygiene,1500000.0,300035.0,20.0
10628,ZWE,98,Humanitarian Crisis in Southern Africa 2002 - ...,CZWE0203,2002-07-01,2003-06-30,2002,NaN,Not specified,NaN,NaN,NaN


In [6]:
merge_keys = ['id', 'code', 'countryCode', 'year', 'name']

df_merged = pd.merge(
    df_funding, 
    df_funding_cluster, 
    on=merge_keys, 
    how='inner',
    suffixes=('_plan_total', '_cluster_specific')
)

numeric_cols = [
    'requirements_plan_total', 'funding_plan_total', 
    'requirements_cluster_specific', 'funding_cluster_specific'
]
for col in numeric_cols:
    df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')


In [7]:
import pandas as pd

def aggregate_needs(year):
    pd.options.display.float_format = '{:,.0f}'.format
    df_needs = pd.read_csv(f"../../data/hpc_hno_{year}.csv", low_memory=False)
    df_needs = df_needs[1:] 

    df_needs["In Need"] = df_needs["In Need"].astype(str).str.replace(',', '')
    df_needs["In Need"] = pd.to_numeric(df_needs["In Need"], errors='coerce').fillna(0)

    df_clean = df_needs[df_needs['Admin 1 PCode'].isna() & df_needs['Category'].isna()]

    df_pivot = pd.pivot_table(
        df_clean, 
        values='In Need', 
        index='Country ISO3', 
        columns='Cluster', 
        aggfunc='sum', 
        fill_value=0
    ).reset_index()

    cluster_cols = [col for col in df_pivot.columns if col != 'Country ISO3']
    df_pivot = df_pivot.rename(columns={col: f"In Need - {col}" for col in cluster_cols})

    return df_pivot


In [15]:
df_needs = aggregate_needs("2024")

cluster_mapping = {
    'In Need - AGR': 'Agriculture',
    'In Need - CCM': 'Camp Coordination / Management', 
    'In Need - CSS': 'Coordination and support services',
    'In Need - EDU': 'Education',
    'In Need - ERY': 'Early Recovery',
    'In Need - FSC': 'Food Security',
    'In Need - HEA': 'Health',
    'In Need - LOG': 'Logistics',
    'In Need - MS':  'Multi-sector',
    'In Need - NUT': 'Nutrition',
    'In Need - PRO': 'Protection',
    'In Need - PRO-CPN': 'Protection - Child Protection',
    'In Need - PRO-GBV': 'Protection - Gender-Based Violence',
    'In Need - PRO-HLP': 'Protection - Housing, Land and Property',
    'In Need - PRO-MIN': 'Protection - Mine Action',
    'In Need - SHL': 'Emergency Shelter and NFI',
    'In Need - WSH': 'Water Sanitation Hygiene',
    'In Need - ALL': 'Total In Need',
    "Country ISO3": "countryCode"
}

df_needs.rename(columns=cluster_mapping, inplace=True)

df_needs = pd.melt(
    df_needs,
    id_vars=['countryCode'],
    value_vars=[c for c in df_needs.columns if c != 'countryCode'],
    var_name='cluster',
    value_name='People_In_Need'
)
df_needs["year"] = 2024
df_needs

,countryCode,cluster,People_In_Need,year
0,AFG,Agriculture,0,2024
1,BFA,Agriculture,0,2024
2,CAF,Agriculture,0,2024
3,CMR,Agriculture,0,2024
4,COD,Agriculture,0,2024
...,...,...,...,...
427,SYR,Water Sanitation Hygiene,"13,563,289",2024
428,TCD,Water Sanitation Hygiene,"3,363,508",2024
429,UKR,Water Sanitation Hygiene,"9,560,978",2024
430,VEN,Water Sanitation Hygiene,"3,387,506",2024


In [19]:
final_analysis = pd.merge(
    df_merged,
    df_needs,
    on=['countryCode', 'cluster', 'year'],
    how='inner' # Inner join ensures we only look at sectors with both documented needs AND a financial plan
)

ValueError: You are trying to merge on str and int64 columns for key 'year'. If you wish to proceed you should use pd.concat

In [17]:
final_analysis

,countryCode,id,name,code,typeId,typeName,startDate_plan_total,endDate_plan_total,year_x,requirements_plan_total,...,percentFunded_plan_total,startDate_cluster_specific,endDate_cluster_specific,clusterCode,cluster,requirements_cluster_specific,funding_cluster_specific,percentFunded_cluster_specific,People_In_Need,year_y
0,AFG,"1,502",Afghanistan Humanitarian Needs and Response Pl...,HAFG26,"2,070",Humanitarian needs and response plan,2026-01-01,2026-12-31,2026,"1,714,181,496",...,13,2026-01-01,2026-12-31,"26,480",Coordination and support services,"24,700,000","1,805,186",7,0,2024
1,AFG,"1,502",Afghanistan Humanitarian Needs and Response Pl...,HAFG26,"2,070",Humanitarian needs and response plan,2026-01-01,2026-12-31,2026,"1,714,181,496",...,13,2026-01-01,2026-12-31,3,Education,"60,040,000","18,138,612",30,"8,030,371",2024
2,AFG,"1,502",Afghanistan Humanitarian Needs and Response Pl...,HAFG26,"2,070",Humanitarian needs and response plan,2026-01-01,2026-12-31,2026,"1,714,181,496",...,13,2026-01-01,2026-12-31,4,Emergency Shelter and NFI,"160,285,975","5,814,964",4,"6,609,590",2024
3,AFG,"1,502",Afghanistan Humanitarian Needs and Response Pl...,HAFG26,"2,070",Humanitarian needs and response plan,2026-01-01,2026-12-31,2026,"1,714,181,496",...,13,2026-01-01,2026-12-31,6,Food Security,"651,101,609","50,221,603",8,"15,823,677",2024
4,AFG,"1,502",Afghanistan Humanitarian Needs and Response Pl...,HAFG26,"2,070",Humanitarian needs and response plan,2026-01-01,2026-12-31,2026,"1,714,181,496",...,13,2026-01-01,2026-12-31,7,Health,"190,768,456","30,587,554",16,"17,936,233",2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4910,YEM,306,Yemen Floods Response Plan (November - April) ...,FYEM0809,5,Flash appeal,2008-11-01,2009-04-30,2008,"5,113,261",...,100,2008-11-01,2009-04-30,4,Emergency Shelter and NFI,"1,671,657","1,519,593",91,"6,726,655",2024
4911,YEM,306,Yemen Floods Response Plan (November - April) ...,FYEM0809,5,Flash appeal,2008-11-01,2009-04-30,2008,"5,113,261",...,100,2008-11-01,2009-04-30,6,Food Security,"1,757,800","1,169,208",67,"17,642,604",2024
4912,YEM,306,Yemen Floods Response Plan (November - April) ...,FYEM0809,5,Flash appeal,2008-11-01,2009-04-30,2008,"5,113,261",...,100,2008-11-01,2009-04-30,7,Health,"3,281,500","1,283,925",39,"17,791,918",2024
4913,YEM,306,Yemen Floods Response Plan (November - April) ...,FYEM0809,5,Flash appeal,2008-11-01,2009-04-30,2008,"5,113,261",...,100,2008-11-01,2009-04-30,10,Protection,"315,583","134,583",43,"16,386,147",2024
